# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yyashkumarsharma23-max/flyrank-internship-machine_learning/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


If a webpage ranks well on Google (Position 1-10) and gets good impressions, but its Click-Through Rate is below 3%, it means users are seeing the link but not clicking it. We should flag this to rewrite the title and meta description to make it more clickable.

**Reason Codes & Actions:**

**Reason Code:** LOW_CTR_HIGH_VISIBILITY

**Action Label:** REWRITE_METADATA

In [4]:
import pandas as pd
import numpy as np

# 1. Loading the dataset
df = pd.read_csv('content_refresh_anonymized.csv')
print(f"Data loaded successfully! Total rows: {len(df)}\n")


# SIGNAL 1: CTR-vs-Position Check

# Bucket positions into pages/top spots using 'avg_position'
pos_bins = [0, 3, 10, 20, 100]
pos_labels = ['Top 3', 'Pos 4-10 (Page 1)', 'Pos 11-20 (Page 2)', 'Pos 21+']
df['position_bucket'] = pd.cut(df['avg_position'], bins=pos_bins, labels=pos_labels)

# Build bucket table with 'n' (count)
signal_1_table = df.groupby('position_bucket', observed=False).agg(
    n=('clicks_90d', 'count'),
    avg_ctr=('ctr', 'mean')
).reset_index()

print("--- Signal 1: CTR-vs-Position ---")
print("Verdict: CONFIRMED (CTR drops as position gets worse)")
display(signal_1_table)

print("\n" + "="*50 + "\n")


# SIGNAL 2: Search Volume Check

# Bucket impressions to see where volume sits using 'impressions_90d'
imp_bins = [-1, 100, 1000, 5000, float('inf')]
imp_labels = ['Low (<100)', 'Medium (100-1K)', 'High (1K-5K)', 'Very High (5K+)']
df['volume_bucket'] = pd.cut(df['impressions_90d'], bins=imp_bins, labels=imp_labels)

# Build bucket table with 'n' (count)
signal_2_table = df.groupby('volume_bucket', observed=False).agg(
    n=('clicks_90d', 'count'),
    avg_clicks=('clicks_90d', 'mean')
).reset_index()

print("--- Signal 2: Search Volume ---")
print("Verdict: CONFIRMED (Higher impressions reliably lead to higher baseline clicks)")
display(signal_2_table)

Data loaded successfully! Total rows: 30000

--- Signal 1: CTR-vs-Position ---
Verdict: CONFIRMED (CTR drops as position gets worse)


,position_bucket,n,avg_ctr
0,Top 3,1141,2.714303
1,Pos 4-10 (Page 1),11842,0.651045
2,Pos 11-20 (Page 2),7273,0.323443
3,Pos 21+,8524,0.211705




--- Signal 2: Search Volume ---
Verdict: CONFIRMED (Higher impressions reliably lead to higher baseline clicks)


,volume_bucket,n,avg_clicks
0,Low (<100),8006,0.146765
1,Medium (100-1K),8485,0.901945
2,High (1K-5K),7359,6.306563
3,Very High (5K+),6150,69.541789


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


In [6]:

# 2. Build the ranked queue

# Copying the dataset to use the dummy data without interfering the original data
scored_df = df.copy()

# By default we assign '0' and 'NONE' to our dataset
scored_df['score'] = 0
scored_df['reason_code'] = 'NO_ACTION'
scored_df['action_label'] = 'NONE'

# Rule: Position should be in between 1-10 and CTR will be less than 3%
rule_condition = (scored_df['avg_position'] <= 10) & (scored_df['ctr'] < 0.03)

# Jin rows mein rule sach hai, wahan score aur labels set karte hain
# Score = impressions_90d (Taki high volume wale links top par rank ho)
scored_df.loc[rule_condition, 'score'] = scored_df.loc[rule_condition, 'impressions_90d']
scored_df.loc[rule_condition, 'reason_code'] = 'LOW_CTR_HIGH_VISIBILITY'
scored_df.loc[rule_condition, 'action_label'] = 'REWRITE_METADATA'

# We have to filter those rows in which 'action_label' is NONE
action_queue = scored_df[scored_df['action_label'] != 'NONE'].copy()

# More the score more will be the rank, top rankers will be placed at the top
action_queue = action_queue.sort_values(by='score', ascending=False)

# For our final output, we have to select those rows like positions from 1-10 or top 10 positions
# For clear understanding of the outputs we create a metric which holds ctr, positions and labels
final_output = action_queue[['content_id', 'score', 'reason_code', 'action_label', 'avg_position', 'ctr', 'impressions_90d']]

print(f"Queue built successfully! Total items flagged for action: {len(final_output)}")

output_path = 'baseline_action_score.csv'
final_output.to_csv(output_path, index=False)

print(f"Ranked queue saved to: {output_path}")

# Displaying first 5 rows to verify the data
display(final_output.head(5))

Queue built successfully! Total items flagged for action: 5899
Ranked queue saved to: baseline_action_score.csv


,content_id,score,reason_code,action_label,avg_position,ctr,impressions_90d
7445,content_c8e9d6ab9013,208678,LOW_CTR_HIGH_VISIBILITY,REWRITE_METADATA,9.7,0.00,208678
27178,content_453722754fea,140079,LOW_CTR_HIGH_VISIBILITY,REWRITE_METADATA,7.6,0.01,140079
3331,content_4a6607efcb46,128068,LOW_CTR_HIGH_VISIBILITY,REWRITE_METADATA,2.2,0.01,128068
15914,content_0919dd345d80,119217,LOW_CTR_HIGH_VISIBILITY,REWRITE_METADATA,7.0,0.02,119217
482,content_39881853ef0c,112434,LOW_CTR_HIGH_VISIBILITY,REWRITE_METADATA,7.2,0.01,112434


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


In [7]:
# Top 20 rows for the manual review
display(final_output.head(20))

,content_id,score,reason_code,action_label,avg_position,ctr,impressions_90d
7445,content_c8e9d6ab9013,208678,LOW_CTR_HIGH_VISIBILITY,REWRITE_METADATA,9.7,0.00,208678
27178,content_453722754fea,140079,LOW_CTR_HIGH_VISIBILITY,REWRITE_METADATA,7.6,0.01,140079
3331,content_4a6607efcb46,128068,LOW_CTR_HIGH_VISIBILITY,REWRITE_METADATA,2.2,0.01,128068
15914,content_0919dd345d80,119217,LOW_CTR_HIGH_VISIBILITY,REWRITE_METADATA,7.0,0.02,119217
482,content_39881853ef0c,112434,LOW_CTR_HIGH_VISIBILITY,REWRITE_METADATA,7.2,0.01,112434
2382,content_65114d89496d,72631,LOW_CTR_HIGH_VISIBILITY,REWRITE_METADATA,6.5,0.02,72631
14226,content_65d9331f55fb,65686,LOW_CTR_HIGH_VISIBILITY,REWRITE_METADATA,7.8,0.02,65686
13631,content_d274ac4158ef,65138,LOW_CTR_HIGH_VISIBILITY,REWRITE_METADATA,6.8,0.01,65138
24866,content_e5f459e737b7,56363,LOW_CTR_HIGH_VISIBILITY,REWRITE_METADATA,5.9,0.01,56363
4589,content_339b357d04c7,46879,LOW_CTR_HIGH_VISIBILITY,REWRITE_METADATA,3.7,0.01,46879


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


I have audited the baseline rule and can confirm there is no data leakage. Our rule is strictly based on historical 90-day aggregate metrics (avg_position, ctr, and impressions_90d).

1. **No Product Flags:** We did not use columns like provider_used or model_used to artificially group or target content.
2. **No Future Windows:** We successfully avoided target window metrics (like impressions_last_30d or trend_pct), ensuring the score is built purely on historical baseline behavior without peeking at recent/future trends.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.